In [1]:
import tensorflow as tf
import numpy as np
import random
import os
import cv2
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, LSTM
from tensorflow.keras.layers import BatchNormalization, TimeDistributed, Input
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import TimeDistributed, Conv2D, MaxPooling2D, Flatten, LSTM, Dense, Dropout, BatchNormalization
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from tensorflow.keras.callbacks import ReduceLROnPlateau, EarlyStopping
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

In [2]:
seed = 42
random.seed(seed)
np.random.seed(seed)
tf.random.set_seed(seed)
os.environ['PYTHONHASHSEED'] = str(seed)

In [3]:
SEQUENCE_LENGTH = 20  # Number of frames per video sequence
FRAME_SIZE = (64, 64)  # Resized frame dimensions
CHANNELS = 3
EPOCHS = 10  # Increased epochs
BATCH_SIZE = 16  # Adjusted batch size
LEARNING_RATE = 0.0001

In [4]:
def load_video_frames(video_path, max_frames=SEQUENCE_LENGTH):
    cap = cv2.VideoCapture(video_path)
    frames = []
    try:
        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame = cv2.resize(frame, FRAME_SIZE)
            frame = frame / 255.0  # Normalize
            frames.append(frame)
            if len(frames) == max_frames:
                break
    finally:
        cap.release()
    while len(frames) < max_frames:
        frames.append(np.zeros((FRAME_SIZE[1], FRAME_SIZE[0], CHANNELS)))
    return np.array(frames)

In [5]:
def load_dataset(directory, label):
    data = []
    for filename in os.listdir(directory):
        video_path = os.path.join(directory, filename)
        frames = load_video_frames(video_path)
        data.append((frames, label))
    return data

In [6]:
train_normal = load_dataset(r"C:\Users\saiku\Downloads\project viedios\train\Normal", 0)  # Add dataset path
train_abnormal = load_dataset(r"C:\Users\saiku\Downloads\project viedios\train\Abnormal", 1)  # Add dataset path
test_normal = load_dataset(r"C:\Users\saiku\Downloads\project viedios\test\Normal", 0)  # Add dataset path
test_abnormal = load_dataset(r"C:\Users\saiku\Downloads\project viedios\test\Abnormal", 1)  # Add dataset path

In [7]:
train_data = train_normal + train_abnormal
random.shuffle(train_data)

X_train = np.array([item[0] for item in train_data])
y_train = np.array([item[1] for item in train_data])

X_test = np.array([item[0] for item in (test_normal + test_abnormal)])
y_test = np.array([item[1] for item in (test_normal + test_abnormal)])

In [8]:
model = Sequential([
    Input(shape=(SEQUENCE_LENGTH, FRAME_SIZE[1], FRAME_SIZE[0], CHANNELS)),  # Explicit input layer

    TimeDistributed(Conv2D(64, (3, 3), activation='relu', padding='same')),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D(2, 2)),

    TimeDistributed(Conv2D(128, (3, 3), activation='relu', padding='same')),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D(2, 2)),

    TimeDistributed(Conv2D(256, (3, 3), activation='relu', padding='same')),
    TimeDistributed(BatchNormalization()),
    TimeDistributed(MaxPooling2D(2, 2)),

    TimeDistributed(Flatten()),
    LSTM(128, return_sequences=False),
    Dropout(0.4),
    Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
    Dropout(0.4),
    Dense(1, activation='sigmoid')
])


In [9]:
model.compile(optimizer=Adam(learning_rate=LEARNING_RATE),
              loss='binary_crossentropy',
              metrics=['accuracy'])

In [10]:
# Adjust EarlyStopping to avoid premature stopping
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

# ReduceLROnPlateau to adjust learning rate dynamically
lr_scheduler = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)


In [ ]:
history = model.fit(X_train, y_train,
                    validation_data=(X_test, y_test),
                    epochs=EPOCHS,
                    batch_size=BATCH_SIZE,
                    callbacks=[lr_scheduler, early_stopping],
                    verbose=1)

Epoch 1/10


In [ ]:
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_acc*100:.2f}%')

In [ ]:
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend()
plt.show()

In [ ]:
y_pred = model.predict(X_test)
y_pred_labels = (y_pred > 0.5).astype(int)
print("\nClassification Report:")
print(classification_report(y_test, y_pred_labels, target_names=['Normal', 'Abnormal']))
cm = confusion_matrix(y_test, y_pred_labels)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Normal', 'Abnormal'],
            yticklabels=['Normal', 'Abnormal'])
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart
import time
import cv2
import numpy as np

def send_alert_email(status, confidence):
    sender_email = "saikumaryenduri7@gmail.com"
    sender_password = "wxpw gzkg jqxf unrx"
    receiver_email = "karthikyelugam09@gmail.com"

    subject = "🔴 ALERT: Abnormal Activity Detected!" if status == "ABNORMAL" else "✅ Normal Activity"
    body = f"Status: {status}\nConfidence: {confidence:.2f}\nCheck the surveillance system immediately!"

    msg = MIMEMultipart()
    msg["From"] = sender_email
    msg["To"] = receiver_email
    msg["Subject"] = subject
    msg.attach(MIMEText(body, "plain"))

    try:
        server = smtplib.SMTP("smtp.gmail.com", 587)
        server.starttls()
        server.login(sender_email, sender_password)
        server.sendmail(sender_email, receiver_email, msg.as_string())
        server.quit()
        print(f"📩 Email alert sent to {receiver_email}")
    except Exception as e:
        print(f"❌ Failed to send email: {e}")

def analyze_live_feed(model, duration=60):
    cap = cv2.VideoCapture(0)  # Use the default camera
    if not cap.isOpened():
        print("Error: Could not open camera")
        return

    frame_buffer = []
    start_time = time.time()
    font = cv2.FONT_HERSHEY_SIMPLEX
    sequence_length = SEQUENCE_LENGTH
    alert_sent = False  # Prevent spamming alerts

    while (time.time() - start_time) < duration:
        ret, frame = cap.read()
        if not ret:
            break

        # Preprocess frame
        resized_frame = cv2.resize(frame, FRAME_SIZE)
        normalized_frame = resized_frame / 255.0
        frame_buffer.append(normalized_frame)

        # Maintain fixed sequence length
        if len(frame_buffer) > sequence_length:
            frame_buffer = frame_buffer[-sequence_length:]

        # Make prediction when buffer is full
        if len(frame_buffer) == sequence_length:
            input_sequence = np.expand_dims(frame_buffer, axis=0)
            prediction = model.predict(input_sequence)[0][0]

            status = "ABNORMAL" if prediction > 0.7 else "NORMAL"
            color = (0, 0, 255) if status == "ABNORMAL" else (0, 255, 0)
            confidence = prediction if status == "ABNORMAL" else 1 - prediction

            # Display prediction
            cv2.putText(frame, f"Status: {status}", (10, 30), font, 0.7, color, 2)
            cv2.putText(frame, f"Confidence: {confidence:.2f}", (10, 70), font, 0.7, color, 2)
            cv2.putText(frame, f"Time remaining: {int(duration - (time.time() - start_time))}s",
                       (10, 110), font, 0.7, (255, 255, 255), 2)

            # Send alert email if abnormal activity detected and not already sent
            if status == "ABNORMAL" and not alert_sent:
                send_alert_email(status, confidence)
                alert_sent = True  # Avoid sending multiple emails

        # Display frame
        cv2.imshow('DETECTION OF ABNORMAL AND NORMAL ACTIVITY', frame)

        # Exit if 'q' is pressed
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Run live analysis for 60 seconds
analyze_live_feed(model, duration=60)